In [ ]:
# ==============================================================================
# Step 1: Mount Google Drive (approve access in the pop-up window)
# ==============================================================================
from google.colab import drive
import os

print("Connecting to Google Drive...")
drive.mount('/content/drive')

# ------------------------------------------------------------------
# Project paths: shared Google Drive folder "dl-quant_project"
# Works for the owner and for teammates who added a shortcut to the
# shared folder in their My Drive (see 00_START_HERE_collaboration).
# ------------------------------------------------------------------
import os, glob

def find_project_dir(name="dl-quant_project"):
    candidates = [
        f"/content/drive/MyDrive/Colab Notebooks/{name}",
        f"/content/drive/MyDrive/{name}",
        *glob.glob(f"/content/drive/MyDrive/*/{name}"),
        *glob.glob(f"/content/drive/Shareddrives/*/{name}"),
    ]
    for p in candidates:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(
        f"Could not find the '{name}' folder in your Google Drive.\n"
        "Fix: Google Drive > Shared with me > right-click 'dl-quant_project' > "
        "Organize > Add shortcut > My Drive. Then re-run this cell.")

PROJECT_DIR  = find_project_dir()
DATA_DIR     = os.path.join(PROJECT_DIR, "DataSource")       # all data files: read AND save here
NOTEBOOK_DIR = os.path.join(PROJECT_DIR, "PythonNoteBooks")  # notebooks
os.makedirs(DATA_DIR, exist_ok=True)
drive_folder = DATA_DIR   # older cells use this name
print("Project folder:", PROJECT_DIR)
print("Data folder:   ", DATA_DIR)

# ==============================================================================
# Step 2: Install dependencies
# ==============================================================================
# Note: huggingface_hub >= 1.0 no longer supports hf_transfer; its default
# downloader is already fast, so no extra settings are needed.
!pip install -q huggingface_hub pandas tqdm

# ==============================================================================
# Step 3: Download the raw FNSPID news file (~23 GB) to Colab's temporary disk
# ==============================================================================
from huggingface_hub import hf_hub_download

print("\nDownloading the raw FNSPID news file (usually 3-5 minutes)...")
try:
    local_file_path = hf_hub_download(
        repo_id="Zihan1004/FNSPID",
        filename="Stock_news/nasdaq_exteral_data.csv",
        repo_type="dataset",
    )
    print(f"Download finished. Temporary path: {local_file_path}")
    print(f"File size: {os.path.getsize(local_file_path) / (1024**3):.2f} GB")
except Exception as e:
    print(f"Download failed, check the network or the Hugging Face repo: {e}")
    raise e

# The file lives in /root/.cache/huggingface and is deleted when the runtime resets.
# Step 4 (filtering GME / AMC / BB) is in the next cell.

正在连接您的 Google Drive，请在弹出的提示框中完成授权...
Mounted at /content/drive

正在配置 Hugging Face 高速下载环境...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(



开始通过高速通道下载原始新闻数据集（通常只需 2-5 分钟）...


Stock_news/nasdaq_exteral_data.csv: reconstructing file:   0%|          |  0.00B / 23.2GB            

Stock_news/nasdaq_exteral_data.csv: downloading bytes:           |  0.00B            

🎉 原始大文件下载成功！临时保存在: /root/.cache/huggingface/hub/datasets--Zihan1004--FNSPID/snapshots/bf9189c41527198897d1af3e17b1a0095279fc45/Stock_news/nasdaq_exteral_data.csv
📦 文件大小: 21.64 GB

开始从本地高效扫描并筛选 {'BB', 'AMC', 'GME'} 的数据...


数据清洗进度: 15549299it [06:27, 40136.83it/s]


❌ 扫描完毕，但未在数据集中匹配到 'GME', 'AMC', 'BB'。


In [ ]:
# ==============================================================================
# Step 4: Filter GME / AMC / BB news and save the result to Google Drive
# ==============================================================================
import pandas as pd
from tqdm import tqdm

target_tickers = {"GME", "AMC", "BB"}
drive_folder = DATA_DIR  # data folder defined in the setup cell
os.makedirs(drive_folder, exist_ok=True)
output_filename = os.path.join(drive_folder, "fnspid_meme_stocks.csv")

# Read 5 rows first to find the ticker column (takes a few seconds)
peek = pd.read_csv(local_file_path, nrows=5)
print("Columns:", peek.columns.tolist())
ticker_col = next((c for c in peek.columns
                   if c.lower() in ("stock_symbol", "ticker", "symbol", "stock")), None)
assert ticker_col, "Ticker column not found, check the column names printed above"
print("Ticker column:", ticker_col)

# Scan the full file in chunks so memory stays low; skip malformed lines
filtered_rows = []
with tqdm(desc="Rows scanned", unit=" rows") as pbar:
    for chunk in pd.read_csv(local_file_path, chunksize=200_000,
                             low_memory=False, on_bad_lines="skip"):
        sym = chunk[ticker_col].astype(str).str.upper().str.strip()
        match = chunk[sym.isin(target_tickers)]
        if not match.empty:
            filtered_rows.append(match)
        pbar.update(len(chunk))

if filtered_rows:
    df_final = pd.concat(filtered_rows, ignore_index=True)
    df_final.to_csv(output_filename, index=False)
    print(f"{len(df_final)} rows saved to {output_filename}")
    print(df_final[ticker_col].str.upper().value_counts())
    # Rows per year, to confirm 2021 is covered
    print(pd.to_datetime(df_final["Date"], errors="coerce", utc=True)
            .dt.year.value_counts().sort_index())
else:
    print("No matches. Check the column names and ticker examples printed above.")

列名: ['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url', 'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']
使用股票代码列: Stock_symbol


扫描行数: 15549299行 [07:22, 35157.40行/s]


共 14772 行，已保存到 /content/drive/MyDrive/Colab Notebooks/dl-quant_project/fnspid_meme_stocks.csv
Stock_symbol
GME    8235
AMC    5057
BB     1480
Name: count, dtype: int64
Date
2009      11
2010     167
2011     248
2012     412
2013     528
2014     535
2015     766
2016     770
2017     858
2018    1210
2019    1208
2020    1110
2021    4351
2022    1591
2023    1007
Name: count, dtype: int64


In [ ]:
# ==============================================================================
# Step 5: Data quality checks
# ==============================================================================
df = df_final.copy()
df["ts"] = pd.to_datetime(df["Date"], errors="coerce", utc=True)
d21 = df[df["ts"].dt.year == 2021]

# 1) Articles per month per ticker in 2021
print(d21.pivot_table(index=d21["ts"].dt.tz_localize(None).dt.to_period("M"),
                      columns="Stock_symbol", values="Url",
                      aggfunc="count", fill_value=0))

# 2) Share of rows with a non-empty article body / summary
for c in ["Article", "Lsa_summary"]:
    print(c, "non-empty share:", round(df[c].notna().mean(), 3))

# 3) Do timestamps include a time of day? (all 00:00 means we cannot split
#    pre-market vs after-close news)
print("Share with a time of day:",
      round(((df["ts"].dt.hour != 0) | (df["ts"].dt.minute != 0)).mean(), 3))

# 4) Duplicates: the same URL listed under several tickers
print("Duplicated URLs:", df["Url"].duplicated().sum())

# 5) A few sample headlines
print(d21.sample(10, random_state=0)[["Date", "Stock_symbol", "Article_title"]].to_string())

Stock_symbol  AMC   BB  GME
ts                         
2021-01       142  103  292
2021-02       223   66  469
2021-03       175   50  238
2021-04        81   14  183
2021-05       128   19   98
2021-06       320  122  227
2021-07       150   21  119
2021-08       137   23   88
2021-09       127   42  126
2021-10        69   17   50
2021-11       101   15   61
2021-12       113   38  104
Article 非空比例: 0.681
Lsa_summary 非空比例: 0.681
有具体时间的比例: 0.001
重复 URL 数: 1204
                         Date Stock_symbol                                                                                               Article_title
8454  2021-01-27 00:00:00 UTC          GME  Pre-Market Most Active for Jan 27, 2021 : AMC, EXPR, BB, NOK, GME, SQQQ, AAPL, CCIV, QQQ, BBBY, TTOO, MSFT
1725  2021-08-02 00:00:00 UTC          AMC                                Want to Be a Millionaire? Don't Let This Investing Mistake Derail Your Plans
7159  2021-06-10 00:00:00 UTC          GME                                      

/tmp/ipykernel_3992/2085765405.py:6: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  print(d21.pivot_table(index=d21["ts"].dt.to_period("M"), columns="Stock_symbol",


In [ ]:
# ==============================================================================
# Step 6: Preprocessing - dedupe, relevance filter, text field, target date
# ==============================================================================
import re
import numpy as np

df = df_final.copy()
df["ticker"] = df["Stock_symbol"].str.upper().str.strip()
# Timestamps are date-only (00:00 UTC), so keep just the calendar date
df["date"] = (pd.to_datetime(df["Date"], errors="coerce", utc=True)
                .dt.tz_localize(None).dt.normalize())
df = df.dropna(subset=["date"])

# 1) Dedupe: keep one row per (ticker, URL)
before = len(df)
df = df.drop_duplicates(subset=["ticker", "Url"])
print(f"Dedupe: {before} -> {len(df)} rows")

# 2) Relevance: how often the company is mentioned in the title and body
patterns = {
    "GME": r"gamestop|\bGME\b",
    "AMC": r"\bAMC\b",
    "BB":  r"blackberry|\bBB\b",   # note: "BB" can also be a credit rating
}

def count_hits(text, pat):
    """Count case-insensitive matches of pat in text (0 for missing text)."""
    return len(re.findall(pat, text, flags=re.IGNORECASE)) if isinstance(text, str) else 0

df["title_hits"] = [count_hits(t, patterns[k]) for t, k in zip(df["Article_title"], df["ticker"])]
df["body_hits"]  = [count_hits(a, patterns[k]) for a, k in zip(df["Article"], df["ticker"])]

# Generic market round-ups / ticker lists are not about one specific stock
df["is_list"] = df["Article_title"].str.contains(
    r"Most Active|STOCKS-|Stocks Making the Biggest Moves", case=False, na=False)

# Rule: company named in the title, or at least 3 mentions in the body,
# and not a generic list / market round-up
df["relevant"] = ((df["title_hits"] > 0) | (df["body_hits"] >= 3)) & ~df["is_list"]
print(df.groupby("ticker")["relevant"].agg(["sum", "mean"]).round(3))

# 3) Text to score: title + first 1,500 characters of the body
#    (title only when the body is missing)
df["text"] = df["Article_title"].fillna("") + ". " + df["Article"].fillna("").str[:1500]

# 4) Target date: the first trading day strictly AFTER the news date (D+1).
#    Timestamps have no time of day, so news dated D may have been published
#    after the close; using D+1 avoids look-ahead bias.
!pip -q install yfinance
import yfinance as yf
px = yf.download(["GME", "AMC", "BB"], start="2009-01-01", end="2024-01-01",
                 auto_adjust=False, progress=False)
trading_days = px.index.normalize().values.astype("datetime64[ns]")
idx = np.searchsorted(trading_days, df["date"].values.astype("datetime64[ns]"), side="right")
df = df[idx < len(trading_days)].copy()
df["target_date"] = trading_days[idx[idx < len(trading_days)]]

# 5) Save the relevant news and the prices to Google Drive
clean = df[df["relevant"]]
out = os.path.join(drive_folder, "fnspid_meme_clean.csv")
clean[["ticker", "date", "target_date", "Article_title", "text",
       "title_hits", "body_hits", "Publisher", "Url"]].to_csv(out, index=False)
px.to_csv(os.path.join(drive_folder, "prices_gme_amc_bb.csv"))
print(f"\nKept {len(clean)} relevant articles, saved to: {out}")

# 6) Spot check: dropped vs kept headlines, to tune the relevance rule
print("\n--- Dropped (sample) ---")
print(df[~df["relevant"]].sample(8, random_state=1)[["ticker", "Article_title"]].to_string())
print("\n--- Kept (sample) ---")
print(clean.sample(8, random_state=1)[["ticker", "Article_title"]].to_string())

Dedupe: 14772 -> 14733 rows
         sum   mean
ticker             
AMC     3229  0.639
BB      1139  0.770
GME     5170  0.630

Kept 9538 relevant articles, saved to: /content/drive/MyDrive/Colab Notebooks/dl-quant_project/fnspid_meme_clean.csv

--- Dropped (sample) ---
      ticker                                                                    Article_title
4956      BB    BUZZ-U.S. STOCKS ON THE MOVE-Methode Electronics, Drive Shack Inc, CarMax Inc
11432    GME                Merging Office Retailers See Short Interest Surge (AVP, ODP, OMX)
10768    GME                                        Benzinga's Option Alert Recap From May 26
6508     GME                  YEARENDER-The Year of the Doge? 2021, crypto's wildest year yet
5480     GME  US STOCKS-S&P, Nasdaq eye higher open on Alphabet boost, payrolls data in focus
12391    AMC             IPic Entertainment: Poor Business Model Targeting A Declining Market
10340    GME   Mid-Afternoon Market Update: Dow Falls 550 Points; BIO-

In [ ]:

drive_folder = DATA_DIR  # data folder defined in the setup cell

# Cleaned news file
clean = pd.read_csv(os.path.join(drive_folder, "fnspid_meme_clean.csv"))
print(clean.columns.tolist())

['ticker', 'date', 'target_date', 'Article_title', 'text', 'title_hits', 'body_hits', 'Publisher', 'Url']
